# Neuro-JEPA structural embeddings — **FUTURE PATH (requires institutional HF access)**

> **This notebook does NOT run end-to-end for this team, by design.** It is an
> honest scaffold for the *real frozen Neuro-JEPA structural embeddings* feeder
> — one of the three interchangeable feeders behind the contract
> (`src/neuroad/contract.py`). The shipped referee already runs on the
> weight-free structural feeders (OASIS, OpenBHB, gated FreeSurfer drop-ins).

## Two honest blockers

**1. Gated weights that auto-deny us.** The Neuro-JEPA weights are HF-gated
behind an **institutional-PI agreement that auto-denies personal / gmail
emails**, and the assistant cannot accept that agreement on anyone's behalf.
A PI-affiliated user must request access, sign in the browser, and paste an HF
token here. The weight-loading cell below is therefore **guarded pseudocode**.

**2. A preprocessing tax.** Even with weights, each raw T1 must be:
FSL FLIRT-registered to **MNI152 1mm**, **skull-stripped**, and
**bias-field corrected** before the frozen encoder sees it. The encoder is
~**122M parameters**, runs **inference-only**, and fits a **Colab free-tier T4**.

The output of this notebook is a **contract `emb` table** (one row per subject,
`emb_0..emb_{D-1}` + the metadata columns), written to CSV. Dropping that CSV
in via `gated.load_gated(path, <dataset>)` replaces any other feeder with **zero
code change** elsewhere.

## 0. Environment (Colab / local)

FSL is the heavy dependency (registration + skull-strip + bias field). On Colab
install via the official `fslinstaller`; the neuro-imaging Python stack is pip.

In [ ]:
# --- install (Colab) ------------------------------------------------------
# FSL (FLIRT / BET / FAST). ~10 min on Colab; skip if already present.
# !wget -q https://fsl.fmrib.ox.ac.uk/fsldownloads/fslconda/releases/fslinstaller.py
# !python fslinstaller.py -d /usr/local/fsl -q
# import os; os.environ['FSLDIR'] = '/usr/local/fsl'; os.environ['PATH'] += ':/usr/local/fsl/bin'

# Python stack for I/O, the encoder, and the contract.
# !pip install -q nibabel torch huggingface_hub numpy pandas

import numpy as np
import pandas as pd
print('env cell — uncomment installs when running on Colab')

## 1. Load the frozen Neuro-JEPA backbone — **GATED (pseudocode)**

`load_backbone_from_hf` is the intended one-call loader. It requires an HF token
from an account **whose institutional-PI access request was approved** — a
personal/gmail account is auto-denied, so this cell is guarded and will not run
for an unaffiliated user.

In [ ]:
HF_TOKEN = None          # <- paste a PI-approved HF token to enable this path
NEUROJEPA_REPO = 'neuro-jepa/neuro-jepa-structural'   # gated repo id (illustrative)
EMBED_DIM = 256          # encoder output width -> becomes emb_0..emb_255


def load_backbone_from_hf(repo_id: str, token: str):
    """Download + instantiate the frozen 122M-param Neuro-JEPA encoder.

    GATED: needs an institutional-PI-approved HF token. Returns an
    inference-only torch module mapping a preprocessed (1,1,182,218,182)
    MNI152-1mm volume -> a (EMBED_DIM,) feature vector.
    """
    if token is None:
        raise PermissionError(
            'Neuro-JEPA weights are HF-gated behind an institutional-PI '
            'agreement that auto-denies personal/gmail emails. Set HF_TOKEN to a '
            'PI-approved token to enable this cell. This is a documented future '
            'path, not a shipped feature.')
    # --- real path (runs only for an approved user) -----------------------
    # from huggingface_hub import hf_hub_download
    # import torch
    # ckpt = hf_hub_download(repo_id, 'encoder.pt', token=token)
    # model = torch.load(ckpt, map_location='cpu').eval()
    # for p in model.parameters():
    #     p.requires_grad_(False)          # frozen
    # return model
    raise NotImplementedError('wire in torch.load once weights are accessible')


try:
    model = load_backbone_from_hf(NEUROJEPA_REPO, HF_TOKEN)
except (PermissionError, NotImplementedError) as e:
    model = None
    print('GATED backbone not loaded (expected for this team):\n ', e)

## 2. Preprocess a raw T1 — the preprocessing tax

FSL FLIRT to MNI152 1mm, skull-strip (BET), bias-field correction (FAST). This
is required whether or not the weights are available; it is the fixed cost of
feeding a raw NIfTI (IXI / MIRIAD / raw OASIS-3) to the frozen encoder.

In [ ]:
import subprocess, os

MNI152_1MM = os.path.join(os.environ.get('FSLDIR', ''),
                          'data/standard/MNI152_T1_1mm_brain.nii.gz')


def preprocess_t1(raw_t1_path: str, workdir: str) -> str:
    """raw T1 -> MNI152-1mm, skull-stripped, bias-corrected volume path.

    Steps (FSL): BET skull-strip -> FAST bias field -> FLIRT to MNI152 1mm.
    Returns the path to the registered, cleaned NIfTI ready for the encoder.
    """
    os.makedirs(workdir, exist_ok=True)
    brain = os.path.join(workdir, 'brain.nii.gz')
    restored = os.path.join(workdir, 'brain_restore.nii.gz')
    mni = os.path.join(workdir, 'brain_mni1mm.nii.gz')
    # 1) skull-strip
    #    subprocess.run(['bet', raw_t1_path, brain, '-R', '-f', '0.5'], check=True)
    # 2) bias-field correction (FAST writes *_restore)
    #    subprocess.run(['fast', '-B', '-o', os.path.join(workdir,'fast'), brain], check=True)
    # 3) affine register to MNI152 1mm
    #    subprocess.run(['flirt', '-in', restored, '-ref', MNI152_1MM,
    #                    '-out', mni, '-dof', '12'], check=True)
    # (commands commented so the notebook imports cleanly without FSL/data.)
    return mni


print('preprocess_t1 defined — uncomment the FSL calls when FSL + a raw T1 are present')

## 3. Inference — one preprocessed volume -> one feature vector

Inference-only forward pass through the frozen encoder. Falls back to a clearly
labelled placeholder vector when the gated weights are absent, so the rest of
the notebook (table assembly + CSV write) can be exercised for shape/contract.

In [ ]:
def embed_volume(model, mni_volume_path: str, dim: int = EMBED_DIM) -> np.ndarray:
    """Preprocessed MNI152-1mm volume -> (dim,) frozen-encoder feature vector."""
    if model is None:
        # GATED: no weights -> emit a labelled placeholder so downstream shape
        # is exercisable. NEVER report a number computed from this as a result.
        return np.full(dim, np.nan, dtype=float)
    # --- real path --------------------------------------------------------
    # import nibabel as nib, torch
    # vol = nib.load(mni_volume_path).get_fdata().astype('float32')
    # vol = (vol - vol.mean()) / (vol.std() + 1e-6)
    # x = torch.from_numpy(vol)[None, None]      # (1,1,X,Y,Z)
    # with torch.no_grad():
    #     feat = model(x).squeeze(0).cpu().numpy()
    # return feat
    raise NotImplementedError('wire in the forward pass once weights are accessible')


print('embed_volume defined')

## 4. Assemble a **contract `emb` table** and write CSV

Loop subjects (raw T1 + a clinical manifest of `dx / age / sex / site / scanner`
/ biomarkers), preprocess, embed, and assemble the frozen contract schema. This
is the same schema the stubs and the weight-free feeders use, so the output CSV
drops into `gated.load_gated` with zero code change.

In [ ]:
import sys
sys.path.insert(0, '../src')          # so `from neuroad import contract` works
from neuroad import contract

# A clinical manifest: one row per subject, path to the raw T1 + the labels the
# encoder cannot know. (Illustrative 2-row manifest; replace with your cohort.)
manifest = pd.DataFrame([
    {'subject_id': 'SUBJ_0001', 't1_path': '/data/subj0001_T1.nii.gz',
     'dx': 'CN',  'age': 71.2, 'sex': 'F', 'site': 'SITE_A',
     'scanner': 'Siemens_Prisma_3T', 'conversion': pd.NA,
     'amyloid': 0, 'p_tau217': 0.18, 'gfap': 88.0, 'nfl': 12.4, 'apoe4': 0},
    {'subject_id': 'SUBJ_0002', 't1_path': '/data/subj0002_T1.nii.gz',
     'dx': 'AD',  'age': 80.4, 'sex': 'M', 'site': 'SITE_B',
     'scanner': 'GE_Discovery_3T', 'conversion': pd.NA,
     'amyloid': 1, 'p_tau217': 0.61, 'gfap': 214.9, 'nfl': 33.2, 'apoe4': 2},
])

vectors = []
for _, row in manifest.iterrows():
    mni = preprocess_t1(row['t1_path'], workdir=f"/tmp/{row['subject_id']}")
    vectors.append(embed_volume(model, mni, dim=EMBED_DIM))
X = np.vstack(vectors)

# emb_0..emb_{D-1} + the contract metadata columns.
frame = contract.make_embedding_frame(X)
frame.insert(0, 'subject_id', manifest['subject_id'].to_numpy())
frame['dx'] = pd.Categorical(manifest['dx'], categories=contract.DX_LEVELS)
frame['conversion'] = pd.array(manifest['conversion'].to_numpy(), dtype='Int8')
frame['age'] = manifest['age'].astype('float64').to_numpy()
frame['sex'] = pd.Categorical(manifest['sex'], categories=contract.SEX_LEVELS)
frame['site'] = pd.Categorical(manifest['site'])
frame['scanner'] = pd.Categorical(manifest['scanner'])
frame['amyloid'] = pd.array(manifest['amyloid'].to_numpy(), dtype='Int8')
frame['p_tau217'] = manifest['p_tau217'].astype('float64').to_numpy()
frame['gfap'] = manifest['gfap'].astype('float64').to_numpy()
frame['nfl'] = manifest['nfl'].astype('float64').to_numpy()
frame['apoe4'] = pd.array(manifest['apoe4'].to_numpy(), dtype='Int8')

# Validate against the frozen contract. With gated weights absent the emb_* are
# all-NaN placeholders (require_embeddings still passes: the columns exist).
contract.validate_table(frame, require_embeddings=True)

out_csv = 'neurojepa_embeddings.csv'
frame.to_csv(out_csv, index=False)
print(f'wrote {out_csv}: {len(frame)} subjects x {EMBED_DIM}-d embeddings')
print('DROP-IN: gated.load_gated("neurojepa_embeddings.csv", <dataset>) — zero code change downstream')
if model is None:
    print('\n[REMINDER] weights were GATED -> emb_* are placeholders. Not a result.')